### Capstone Project-5: Multi-Agent Research Analyst - Building an Agentic AI System with LangGraph

#### 1.1 Import all the required libraries, packages

In [3]:
import os
import json
import time
import re
import ast
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Callable
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphInterrupt
from langchain_tavily import TavilySearch, TavilyExtract
from groq import Groq

##### List Available Models from groq
- Groq : Open Source platform which gives free inference LLM models.

In [21]:
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

for model in sorted(client.models.list().data, key=lambda x: x.id):
    print(model.id)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.6-27b
whisper-large-v3
whisper-large-v3-turbo


In [5]:

def extract_text_from_response(response) -> str:
    """
    Extract text from LLM response, handling different formats.
    
    - Groq/OpenAI: response.content = "string"
    - Gemini: response.content = [{"type": "text", "text": "string"}, ...]
    """
    content = response.content
    
    # Case 1: Already a string
    if isinstance(content, str):
        return content
    
    # Case 2: List of content blocks (Gemini format)
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict):
                if item.get("type") == "text":
                    text_parts.append(item.get("text", ""))
                elif "text" in item:
                    text_parts.append(item["text"])
        return "\n".join(text_parts)
    
    # Case 3: Fallback
    return str(content)


# ============================================================================
# NOW YOUR LLM INSTANCES
# ============================================================================

llm_quality = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    max_tokens=8192,
    temperature=0.1
)

llm_tools = ChatGroq(
    model="qwen/qwen3.6-27b",
    api_key=os.getenv("GROQ_API_KEY"),
    max_tokens=8192,
    temperature=0.1
)


#### Agent Model Details : 
- Rate Limit : 8k Token / Min
-  120 B Weights : Trained On large Corpus of data.

In [24]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key=os.environ["GROQ_API_KEY"])

#### 1.2 Define Tools
##### Tools: Tools is a function or method which takes defined inputs/arguments and returns a result or performs an action.. Tools Acts as a hands which perform some actions(get data, extract data, hitting API requests) in external apps or systems.
- web_search() : This is the tool which is used to get the latest Details from the internet.
Why use it ? Because LLMS are trained till a specific date eg: GPT 5 Knowledge/Data cutoff is Dec 31st 2025
- Calculation() : This is used to do math calculations LLMS are bad at math calculation because they are next word predictors mainly.
- Extract_context() : Extract the full text content from a web page.
- wikipedia() : Get Details from Wikipedia Website.
- yfinace() : To get the stock details.

#### Wikipedia Tool: INTENTIONALLY REMOVED.
- The underlying wikipedia Python library has a bug
- Tavily web_search is a required tool and searches the whole internet (including Wikipedia). It makes the Wikipedia tool redundant.

In [6]:
"""
Tool Layer for the Multi-Agent Research Analyst

Each tool is a function decorated with @tool.
The docstring is NOT documentation for humans - it's PROMPT for the LLM.
Write it as if you're explaining to a smart assistant who has never seen this tool.
"""


TAVILY_KEY = os.getenv("TAVILY_KEY")


@tool
def web_search(query: str) -> str:  # <-- CHANGED: Removed max_results, changed return to str
    """
    Search the web for current information about a topic.
    
    Use this tool when you need to:
    - Find recent news, articles, or reports
    - Get up-to-date information (post-training cutoff)
    - Discover multiple perspectives on a topic
    - Find specific data points or statistics
    
    Args:
        query: A focused search query. Be specific — "Tata Motors EV sales 2024" 
               is better than "Tata Motors"
    
    Returns:
        A formatted text string containing numbered search results with titles, URLs, and snippets.
    """
    # 1. Force pass the API key
    api_key = os.getenv("TAVILY_KEY") # Keeping your exact variable name
    
    # 2. Initialize (Hardcoded to 5 so the LLM never sees it)
    search_tool = TavilySearch(max_results=5, tavily_api_key=api_key)
    
    # 3. Invoke the search
    response_dict = search_tool.invoke(query)
    
    # 4. Extract the actual list from inside the dictionary
    if isinstance(response_dict, dict):
        raw_results = response_dict.get("results", [])
    else:
        raw_results = response_dict 
        
    # 5. Format as a clean STRING (LLMs read text better than Python dicts)
    if not raw_results:
        return f"No results found for '{query}'"
        
    output_lines = []
    for i, item in enumerate(raw_results, 1):
        title = item.get("title", "No title")
        url = item.get("url", "")
        content = item.get("content", "No content available")[:200] # Truncated to save tokens
        output_lines.append(f"{i}. {title}\n   URL: {url}\n   Snippet: {content}")
        
    return "\n\n".join(output_lines)


@tool
def extract_page(url: str) -> str:
    """
    Extract the full text content from a web page.
    
    Args:
        url: The full URL of the page to extract
    
    Returns:
        The extracted text content of the page
    """
    # ← NEW: Skip PDFs entirely
    if url.lower().endswith('.pdf'):
        return "Error: This is a PDF file which cannot be extracted. Please search for an HTML version or find an alternative source."
    
    try:
        api_key = os.getenv("TAVILY_KEY")
        extractor = TavilyExtract(depth="advanced", tavily_api_key=api_key)
        
        results = extractor.invoke({"urls": [url]})
        
        if results and len(results) > 0:
            content = results[0].get("raw_content", "")
            if content:
                if len(content) > 15000:
                    content = content[:15000] + "\n\n[CONTENT TRUNCATED]"
                return content
            return "Error: No content extracted from this page."
        return "Error: Extraction returned no results."
    
    except Exception as e:
        return f"Error extracting page: {str(e)}"


# ============================================================================
# TOOL 3: SAFE CALCULATOR
# ============================================================================

@tool
def calculator(expression: str) -> str:
    """
    Perform safe arithmetic calculations.
    
    Use this tool when you need to:
    - Calculate percentages, growth rates, or ratios
    - Add, subtract, multiply, or divide numbers found in research
    - Convert between units
    - Compare numerical values
    
    This tool ONLY supports basic arithmetic: +, -, *, /, **, (), and math functions.
    It does NOT support variable assignment, imports, or any code execution.
    
    Args:
        expression: A mathematical expression as a string.
                   Examples: "15000 * 0.12", "(4500 - 3800) / 3800 * 100"
    
    Returns:
        The calculated result as a string, or an error message
    
    Example:
        calculator("(4500 - 3800) / 3800 * 100")  # Growth rate calculation
        → "18.421052631578945"
    """
    # Whitelist of safe operations
    allowed_names = {
        'abs': abs,
        'round': round,
        'min': min,
        'max': max,
        'sum': sum,
        'pow': pow,
        'len': len,
    }
    
    try:
        # Parse the expression into an AST
        tree = ast.parse(expression, mode='eval')
        
        # Walk the AST and check for disallowed operations
        for node in ast.walk(tree):
            # Disallow function calls (except our whitelisted ones)
            if isinstance(node, ast.Call):
                if not isinstance(node.func, ast.Name) or node.func.id not in allowed_names:
                    raise ValueError(f"Function call not allowed: {ast.dump(node)}")
            
            # Disallow attribute access (no obj.method())
            if isinstance(node, ast.Attribute):
                raise ValueError(f"Attribute access not allowed: {ast.dump(node)}")
            
            # Disallow imports
            if isinstance(node, (ast.Import, ast.ImportFrom)):
                raise ValueError("Imports not allowed")
        
        # Compile and evaluate safely
        code = compile(tree, '<string>', 'eval')
        result = eval(code, {"__builtins__": {}}, allowed_names)
        
        return str(result)
    
    except SyntaxError as e:
        return f"Syntax error in expression: {e}"
    except ValueError as e:
        return f"Security error: {e}"
    except Exception as e:
        return f"Calculation error: {e}"


# ============================================================================
# TOOL 4: WIKIPEDIA (Optional but useful for background)
# ============================================================================

@tool  
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia for background information and definitions.
    
    Use this tool when:
    - You need basic definitions or historical context
    - You want to understand a concept before searching for recent info
    - The topic is well-established and likely has a good Wikipedia article
    
    DO NOT use this tool for:
    - Recent news or current events (use web_search instead)
    - Topics that are too niche for Wikipedia
    
    Args:
        query: Search term to look up on Wikipedia
    
    Returns:
        A summary of the Wikipedia article, or an error message
    
    Example:
        wikipedia_search("BYD Company")
        → "BYD Company is a Chinese manufacturing company..."
    """
    try:
        from wikipedia import summary as wiki_summary
        result = wiki_summary(query, sentences=8)
        return result
    except Exception as e:
        return f"Wikipedia search failed: {str(e)}. Try web_search for this topic."


# ============================================================================
# EXPORT ALL TOOLS
# ============================================================================

# This is the list of tools we'll give to researcher agents
RESEARCHER_TOOLS = [web_search, extract_page, calculator]

# Full tool set (including optional ones)
ALL_TOOLS = [web_search, extract_page, calculator, wikipedia_search]


@tool
def get_financials(ticker: str, metric: str = "revenue") -> str:
    """
    Fetch recent financial data for a publicly traded company.
    
    Use this tool when you need:
    - Current stock price or market cap
    - Recent revenue or profit margins
    - Financial context for investment analysis
    
    Args:
        ticker: The stock ticker symbol (e.g., "TSLA", "BYDDF", "TTM")
        metric: The data to fetch. Options: "price", "revenue", "net_income", "market_cap"
    
    Returns:
        A string summarizing the requested financial metric.
    """
    try:
        import yfinance as yf
        stock = yf.Ticker(ticker)
        
        if metric == "price":
            data = stock.history(period="1mo")
            if not data.empty:
                return f"Current price for {ticker}: ${data['Close'].iloc[-1]:.2f}"
            return f"Could not fetch price for {ticker}"
            
        elif metric == "market_cap":
            info = stock.info
            cap = info.get("marketCap", 0)
            if cap:
                return f"Market Cap for {ticker}: ${cap / 1e9:.2f} Billion"
            return f"Market cap data not available for {ticker}"
            
        elif metric == "revenue":
            info = stock.info
            return f"Total Revenue for {ticker}: {info.get('totalRevenue', 'N/A')}"
            
        else:
            return f"Metric '{metric}' not supported. Use 'price', 'revenue', or 'market_cap'."
            
    except Exception as e:
        return f"Error fetching financial data for {ticker}: {str(e)}"

#### Testing the web_search tool

In [ ]:
result = web_search.invoke({
    "query": "Compare the current EV strategies of Tata Motors and BYD"
})

print(result)

1. Indian EV makers beat Tesla, BYD in global efficiency race - Rest of World
   URL: https://restofworld.org/2026/indian-evs-beat-tesla-byd-icct-efficiency-ranking
   Snippet: Tata Motors topped a global ranking of the most efficient battery EVs for 2025, released by the International Council on Clean Transportation. The vehicles in the company’s fleet on average consumed 1

2. Tata attacks India market with $7,000 EV, backed by protectionist policies - Nikkei Asia
   URL: https://asia.nikkei.com/business/automobiles/electric-vehicles/tata-attacks-india-market-with-7-000-ev-backed-by-protectionist-policies
   Snippet: Mexico bets on homegrown electric bus to curb China reliance

- Electric vehicles#### China EVs close in on Japan automakers' Australia stronghold, led by BYD

  China EVs close in on Japan automakers

3. Instagram
   URL: https://www.instagram.com/reel/Db5bDpZglcY
   Snippet: outlook\_business

Why Tata Is Betting Big On EVs Over Hybrids | Outlook Business   
  
In this

In [17]:
response = llm.invoke(
    "What are the current EV strategies of Tata Motors and BYD?"
)

print(response.content)

**Electric‑Vehicle (EV) Strategies – Tata Motors (India) vs. BYD (China & Global)**  
*Status: August 2026*  

---

## 1. Tata Motors (India & Emerging Markets)

| Pillar | What Tata is Doing | Why It Matters |
|--------|-------------------|----------------|
| **Product Portfolio** | • **Tata Nexon EV** (sub‑compact SUV) – > 200 kWh sales per month, price ≈ ₹14 lakh (US$ 170). <br>• **Tata Altroz EV** (hatchback) – launched Q2 2025, targeted at metro commuters, price ≈ ₹12 lakh. <br>• **Tata Curvv EV** (mid‑size SUV) – slated for Q4 2025, 500 km WLTP range, 75 kWh battery. <br>• **Tata Passenger‑Van EV** (commercial) – 2024‑2026 rollout for fleet operators, 300 km range, 60 kWh pack. | Covers the three most price‑sensitive segments in India (compact, mid‑size, commercial) and creates a “ladder” that lets customers stay within the Tata brand as they upgrade. |
| **Platform & Architecture** | • **E‑Space Architecture** – a dedicated skateboard chassis (flat battery pack, modular motor) u

In [ ]:
llm_with_tools = llm.bind_tools([web_search, calculator])

response = llm_with_tools.invoke(
    "Compare the current EV strategies of Tata Motors and BYD. Use web_search."
)

#### Tool Testing 

In [4]:
print("Testing Calculator : ")
print(calculator.invoke({"expression" : "(500 - 400) / 400 * 100"}), "% growth")

Testing Calculator : 
25.0 % growth


In [ ]:
print("Testing Web Search: ")
print(web_search.invoke({"query" : "Check Details about Energy Transition in india ?"}))

Testing Web Search: 
1. How India is Powering the Global Energy Transition
   URL: https://www.investindia.gov.in/team-india-blogs/how-india-powering-global-energy-transition
   Snippet: India's energy transition is anchored in long-term policy clarity. The country has committed to achieving 500 GW of non-fossil fuel capacity by 2030 and reaching net-zero emissions by 2070.( By Novemb

2. India - Energy Transitions Commission
   URL: https://www.energy-transitions.org/region/india
   Snippet: Over the last few years, ETC India – based in Delhi and run by The Energy and Resources Institute – has become an authoritative voice on India’s energy transition. It has influenced the upward revisio

3. Navigating the energy transition in India: challenges and opportunities towards sustainable energy goal
   URL: https://www.sciencedirect.com/science/article/pii/S2588912525000190
   Snippet: India holds a pivotal position in the evolving global energy landscape, presenting immense opportunities 

In [7]:
print("Testing wikipedia:")
print(wikipedia_search.invoke({"query" : "Opputunities in energy transition in india investing and jobs?"}))

Testing wikipedia:
Wikipedia search failed: Page id "opportunities in energy transition in india investing and jobs " does not match any pages. Try another id!. Try web_search for this topic.


#### State : Data or Input we pass to agent graph each step in the agent graph will update each item/attribute/variable of the state.
#### why state ?
- Because Agents nodes take query as input and give the reasoned or logical answer as output. As the functions varibales scope is local to function and will be vanished in RAM So we need to save the state in some place that is the reason state is used. 

#### GPT Answer : 
- State is the shared data/context of an agent graph. Each node reads the information it needs from the state and returns updates to the state. This allows different nodes to communicate and progressively build the result as the graph executes.

In [7]:
"""
STEP 4: SHARED STATE DEFINITION

This is the MOST IMPORTANT file in multi-agent system.
Every agent reads from and writes to this state.

KEY CONCEPTS:
1. TypedDict: Defines the "shape" of our whiteboard (what columns exist).
2. Annotated[X, reducer]: Tells LangGraph HOW to handle multiple writes to column X.
"""

class ResearchState(TypedDict):
    """
    The shared whiteboard for our multi-agent system.
    """
    
    # ========================================================================
    # INPUT (Written once by the user, read by everyone)
    # ========================================================================
    question: str
    """The original research question from the user."""
    
    
    # ========================================================================
    # PLANNING PHASE (Written by Planner, read by everyone)
    # ========================================================================
    plan: List[str]
    """
    List of sub-questions (e.g., ["What is Tata's market share?", "What is BYD's strategy?"]).
    NO REDUCER NEEDED: Only the Planner writes to this, so there's no collision risk.
    """
    
    
    # ========================================================================
    # RESEARCH PHASE (Written by PARALLEL Researchers) - NEEDS REDUCER!
    # ========================================================================
    # Findings now include sources
    findings: Annotated[List[Dict[str, Any]], operator.add]
    """
    Each finding is now:
    {
        "researcher_id": 1,
        "sub_question": "...",
        "answer": "...",
        "sources": [           ← NEW
            {"index": 1, "title": "...", "url": "..."},
            {"index": 2, "title": "...", "url": "..."}
        ]
    }
    """
    
    
    # ========================================================================
    # SYNTHESIS PHASE (Written by Synthesiser, read by Critic)
    # ========================================================================
    draft: str
    """The combined research brief written by the Synthesiser. 
    NO REDUCER: Only one writer at this stage."""
    
    references: List[Dict[str, str]]
    """List of citations [{'index': 1, 'url': '...'}]. 
    NO REDUCER: Only one writer."""
    
    
    # ========================================================================
    # CRITIQUE PHASE (Written by Critic, used for routing)
    # ========================================================================
    critique: str
    """The Critic's feedback. Either 'APPROVED' or a list of gaps."""
    
    iterations: int
    """How many times we've gone through the Critic loop. Used to prevent infinite loops."""
    
    approved: bool
    """True if the Critic approves the draft. False otherwise."""

    references: List[Dict[str, Any]]
    """Store the urls for sources"""




#### Nodes

- **Node:** A node is a unit of work in an agent graph. Programmatically, it is typically a function that receives the current graph state, performs an operation such as calling an LLM, tool, API, database, or Python logic, and returns updates to the state.

In [25]:
# Agents Code

"""
Specialist Sub-Agents for the Multi-Agent Research System.

This module defines the core logic for each node in the LangGraph workflow.
Each function takes the current ResearchState, performs a specific task,
and returns a partial dictionary to update the shared state.
"""

# Maximum allowed iterations for the Critic loop to prevent infinite runs
MAX_CRITIC_ITERATIONS = 2


# ============================================================================
# NODE 1: PLANNER
# ============================================================================

PLANNER_SYSTEM_PROMPT = """You are an expert research planner. Your task is to decompose a complex research question into 3 to 5 distinct, searchable sub-questions.

RULES:
1. Each sub-question must be specific enough to be answered by a single web search.
2. Sub-questions must not overlap in scope.
3. Together, the sub-questions must comprehensively cover the original question.
4. Respond ONLY with a valid JSON array of strings. No markdown, no explanation.

EXAMPLE INPUT: What is the investment case for green hydrogen in India?
EXAMPLE OUTPUT: ["What are the current government subsidies for green hydrogen in India?", "Who are the key private players investing in Indian green hydrogen?", "What are the main risks and cost challenges for green hydrogen production in India?"]
"""

def planner_node(state: ResearchState) -> dict:
    """
    Decomposes the main question into a list of sub-questions.
    
    Args:
        state: The current shared state containing the user's question.
        
    Returns:
        A dictionary updating the 'plan', 'iterations', and 'approved' keys.
    """
    print(f"[PLANNER] Decomposing question: {state['question']}")
    
    response = llm.invoke(
        [HumanMessage(content=f"{PLANNER_SYSTEM_PROMPT}\n\nQUESTION: {state['question']}")]
    )
    
    raw_content = response.content.strip()
    
    # Robust JSON extraction: LLMs sometimes wrap JSON in markdown code blocks
    json_match = re.search(r"\[.*\]", raw_content, re.DOTALL)
    if json_match:
        try:
            plan = json.loads(json_match.group(0))
            if not isinstance(plan, list):
                plan = [plan]
        except json.JSONDecodeError:
            plan = _generate_fallback_plan(state["question"])
    else:
        plan = _generate_fallback_plan(state["question"])
        
    # Ensure we don't exceed 5 sub-questions
    plan = plan[:5]
    print(f"[PLANNER] Generated {len(plan)} sub-questions.")
    
    return {
        "plan": plan,
        "iterations": 0,
        "approved": False
    }


def _generate_fallback_plan(question: str) -> List[str]:
    """Generates a generic fallback plan if JSON parsing fails."""
    return [
        f"What is the current state of {question}?",
        f"What are the key drivers or causes related to {question}?",
        f"What are the future projections or solutions for {question}?"
    ]


# ============================================================================
# NODE 2: RESEARCHER (Agent Node with Tools)
# ============================================================================

RESEARCHER_PROMPT_HEADER = """You are a focused research specialist. Answer the sub-question using the available tools.

Available Tools:
- web_search: Takes a {{"query": "string"}} to search the web.
- extract_page: Takes a {{"url": "string"}} to read a full webpage.
- calculator: Takes an {{"expression": "string"}} to do math.

You MUST use the following format strictly:
Question: {question}
Thought: I need to search for...
Action: web_search
Action Input: {{"query": "my search query"}}
Observation: [Tool output will appear here]
... (repeat as needed)
Thought: I now have the final answer.
Final Answer: [Write your detailed 2-3 paragraph answer here, citing sources like [1], [2]]"""

def _run_manual_react_loop(sub_question: str) -> tuple[str, list[dict]]:
    """
    ReAct loop that returns BOTH the answer AND the collected sources.
    
    Returns:
        tuple: (answer_text, list_of_sources_with_urls)
    """
    tools_map = {
        "web_search": web_search,
        "extract_page": extract_page,
        "calculator": calculator
    }
    tools_list = [web_search, extract_page, calculator]
    
    system_prompt = """You are a focused research specialist. 
Answer the sub-question by using the available tools to search for information.

Rules:
1. Use web_search to find relevant information
2. Use extract_page if you need more detail from a URL
3. After gathering information, provide a detailed 2-3 paragraph answer
4. Cite sources using [1], [2], etc. - match these to the ORDER you received search results
5. Be specific with data, numbers, and quotes"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Research this question: {sub_question}"}
    ]
    
    llm_with_tools = llm.bind_tools(tools_list, tool_choice="required")
    
    # ← NEW: Track sources as we go
    collected_sources = []  # Will store: {"index": 1, "title": "...", "url": "..."}
    source_index = 0
    
    for step in range(6):
        for attempt in range(5):
            try:
                response = llm_with_tools.invoke(messages)
                break
            except Exception as e:
                if "429" in str(e) or "rate_limit" in str(e):
                    wait_time = 10 * (attempt + 1)
                    print(f"[RATE LIMIT] Waiting {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    raise e
        else:
            return "Error: Failed after retries.", collected_sources
        
        if response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                
                print(f"  [Tool] {tool_name}({tool_args.get('query', tool_args)})")
                
                if tool_name in tools_map:
                    try:
                        result = tools_map[tool_name].invoke(tool_args)
                        
                        # ← NEW: Parse the result to extract URLs
                        if tool_name == "web_search" and isinstance(result, str):
                            # Parse the formatted output to extract sources
                            for line in result.split("\n"):
                                if line.strip().startswith("URL:"):
                                    url = line.replace("URL:", "").strip()
                                    # Find the title (previous line)
                                    lines = result.split("\n")
                                    for i, l in enumerate(lines):
                                        if l.strip().startswith("URL:") and lines[i-1].strip():
                                            title = lines[i-1].strip().lstrip("0123456789. ")
                                            source_index += 1
                                            collected_sources.append({
                                                "index": source_index,
                                                "title": title,
                                                "url": url
                                            })
                                            break
                        
                        if len(result) > 3000:
                            result = result[:3000] + "\n[...truncated...]"
                    except Exception as e:
                        result = f"Error: {str(e)}"
                else:
                    result = f"Error: Unknown tool '{tool_name}'"
                
                messages.append({"role": "assistant", "content": response.content, "tool_calls": [tool_call]})
                messages.append({
                    "role": "tool", 
                    "tool_call_id": tool_call["id"],
                    "content": result
                })
        else:
            return response.content if response.content else "No answer generated.", collected_sources
    
    return "Error: Max tool loops reached.", collected_sources

def create_researcher_node(researcher_id: int) -> Callable:
    """Factory function to create a researcher node."""
    
    def researcher_node(state: ResearchState) -> dict:
        plan = state.get("plan", [])
        
        if researcher_id >= len(plan):
            print(f"[RESEARCHER {researcher_id + 1}] No sub-question assigned. Skipping.")
            return {"findings": []}
            
        sub_question = plan[researcher_id]
        print(f"[RESEARCHER {researcher_id + 1}] Investigating: {sub_question}")
        
        # ← CHANGED: Now unpacks tuple
        answer, sources = _run_manual_react_loop(sub_question)

        
        # ← DEBUG: Print what we captured
        print(f"[RESEARCHER {researcher_id + 1}] Sources captured:")
        for src in sources:
            print(f"    [{src['index']}] {src['title'][:50]}... -> {src['url'][:60]}...")
        
     
        finding = {
            "researcher_id": researcher_id + 1,
            "sub_question": sub_question,
            "answer": answer,
            "sources": sources  # ← NEW: URLs preserved here!
        }
        
        print(f"[RESEARCHER {researcher_id + 1}] Completed. Found {len(sources)} sources.")
        return {"findings": [finding]}
        
    return researcher_node


# ============================================================================
# NODE 3: SYNTHESISER
# ============================================================================

SYNTHESISER_PROMPT = """You are an expert technical writer. Synthesize the research findings into a cohesive brief.

ORIGINAL QUESTION: {question}

RESEARCH FINDINGS:
{findings_text}

CRITICAL CITATION RULES:
1. Each finding has numbered sources. Use [1], [2], etc. to cite them.
2. When you write a claim, IMMEDIATELY cite which source it came from.
3. You MUST include a "## References" section at the end.
4. In References, list EACH source with its FULL URL - format:
   [1] "Title of Source" - https://full-url.com
   [2] "Title of Source" - https://full-url.com
5. NEVER use vague references like "search results" or "sources say"
6. If a finding has no sources, say "Unsourced claim" and don't use [X] notation

INSTRUCTIONS:
1. Weave findings into a logical narrative (don't just list them)
2. Use clear ## headings
3. Be objective - acknowledge contradictions if findings disagree
4. End with ## References section with FULL URLs"""

def synthesiser_node(state: ResearchState) -> dict:
    """Combines all findings into a single drafted brief."""
    print("[SYNTHESISER] Writing research brief...")
    
    findings_text = ""
    all_references = []  # ← NEW: Collect all references
    
    for i, finding in enumerate(state.get("findings", []), 1):
        findings_text += f"--- FINDING {i} ---\n"
        findings_text += f"Sub-Question: {finding['sub_question']}\n"
        findings_text += f"Answer: {finding['answer']}\n"
        
        # ← NEW: Include sources with URLs in the text
        if finding.get('sources'):
            findings_text += f"Sources for this finding:\n"
            for src in finding['sources']:
                findings_text += f"  [{src['index']}] \"{src['title']}\" - {src['url']}\n"
                all_references.append(src)  # Collect for final reference list
        
        findings_text += "\n"
    
    response = llm.invoke([
        HumanMessage(content=SYNTHESISER_PROMPT.format(
            question=state["question"],
            findings_text=findings_text
        ))
    ])
    
    draft = response.content
    
    # ← NEW: Store references with actual URLs
    references = [
        {"id": src["index"], "title": src["title"], "url": src["url"]}
        for src in all_references
    ]
    
    print(f"[SYNTHESISER] Draft complete. {len(references)} references captured.")
    return {"draft": draft, "references": references}


# ============================================================================
# NODE 4: CRITIC
# ============================================================================

CRITIC_PROMPT = """You are a rigorous quality control editor. Evaluate the following research brief.

ORIGINAL QUESTION: {question}

RESEARCH BRIEF:
{draft}

EVALUATION CRITERIA:
1. Coverage: Does it answer all parts of the original question?
2. Faithfulness: Are all claims supported by the citations? (No hallucinations?)
3. Structure: Is it well-organized and readable?

OUTPUT INSTRUCTIONS:
If the brief is excellent and requires no changes, respond with exactly one word: APPROVED
If the brief has gaps, missing citations, or hallucinations, respond with: 
NEEDS_REVISION: [Provide a specific, actionable list of what is missing or wrong]"""

def critic_node(state: ResearchState) -> dict:
    """
    Evaluates the draft and decides if it is ready for delivery.
    
    Args:
        state: The current shared state containing the draft.
        
    Returns:
        A dictionary updating the 'critique', 'approved', and 'iterations' keys.
    """
    current_iterations = state.get("iterations", 0) + 1
    print(f"[CRITIC] Evaluating draft (Iteration {current_iterations}/{MAX_CRITIC_ITERATIONS})...")
    
    response = llm.invoke(
        [HumanMessage(content=CRITIC_PROMPT.format(
            question=state["question"],
            draft=state["draft"]
        ))]
    )
    
    critique = response.content.strip()
    is_approved = "APPROVED" in critique.upper() and "NEEDS_REVISION" not in critique.upper()
    
    if is_approved:
        print("[CRITIC] Decision: APPROVED")
    else:
        print(f"[CRITIC] Decision: NEEDS REVISION")
        print(f"[CRITIC] Feedback: {critique[:200]}...")
        
    return {
        "critique": critique,
        "approved": is_approved,
        "iterations": current_iterations
    }


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def _extract_final_answer(messages: List[Any]) -> str:
    """
    Parses the LangGraph message history to find the final text response.
    """
    for msg in reversed(messages):
        if hasattr(msg, 'type') and msg.type == 'ai':
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                continue # Skip intermediate tool-calling messages
            if hasattr(msg, 'content') and msg.content.strip():
                return msg.content
    return "Error: Agent failed to produce a final text answer."


def should_continue(state: ResearchState) -> str:
    """
    Conditional edge routing function.
    Determines if the workflow should loop back to the synthesiser or end.
    
    Args:
        state: The current shared state.
        
    Returns:
        "revise" if the draft needs work and we haven't hit the max iterations.
        "end" if approved or max iterations reached.
    """
    if state.get("approved", False):
        return "end"
        
    if state.get("iterations", 0) >= MAX_CRITIC_ITERATIONS:
        print("[ROUTER] Max iterations reached. Proceeding with current draft.")
        return "end"
        
    return "revise"

In [26]:
# Graph Code

"""
LangGraph Orchestration Layer.

This module defines the state graph structure. It connects the specialist 
agents (nodes) using edges and conditional routing to execute the research pipeline.
"""

def build_research_graph(checkpointer=None, enable_interrupt=True) -> StateGraph:
    """
    Constructs and compiles the multi-agent research workflow.
    
    Architecture Patterns Used:
    - Planner-Executor: Planner creates the plan, graph executes it.
    - Orchestrator-Workers: Graph fans out to parallel researchers.
    - Evaluator-Optimizer: Critic loops the draft back to synthesiser.
    
    Args:
        checkpointer: Optional InMemorySaver instance for state persistence.
        enable_interrupt: If True, pauses before researchers for Human-in-the-Loop.
                         Set to False for automated web deployments (Streamlit).
    
    Returns:
        A compiled LangGraph StateGraph ready for invocation.
    """
    
    # Initialize the graph with our strict state schema
    workflow = StateGraph(ResearchState)
    
    # ========================================================================
    # 1. ADD NODES
    # ========================================================================
    
    workflow.add_node("planner", planner_node)
    workflow.add_node("synthesiser", synthesiser_node)
    workflow.add_node("critic", critic_node)
    
    workflow.add_node("researcher_1", create_researcher_node(0))
    workflow.add_node("researcher_2", create_researcher_node(1))
    workflow.add_node("researcher_3", create_researcher_node(2))
    
    # ========================================================================
    # 2. ADD EDGES (The Wiring)
    # ========================================================================
    
    workflow.add_edge(START, "planner")
    
    workflow.add_edge("planner", "researcher_1")
    workflow.add_edge("planner", "researcher_2")
    workflow.add_edge("planner", "researcher_3")
    
    workflow.add_edge("researcher_1", "synthesiser")
    workflow.add_edge("researcher_2", "synthesiser")
    workflow.add_edge("researcher_3", "synthesiser")
    
    workflow.add_edge("synthesiser", "critic")
    
    workflow.add_conditional_edges(
        "critic",
        should_continue,
        {
            "revise": "synthesiser",
            "end": END
        }
    )
    
    # ========================================================================
    # 3. COMPILE WITH PERSISTENCE AND CONDITIONAL INTERRUPT
    # ========================================================================
    
    if checkpointer is None:
        checkpointer = InMemorySaver()
        
    compile_args = {"checkpointer": checkpointer}
    
    if enable_interrupt:
        compile_args["interrupt_before"] = ["researcher_1"]
        
    return workflow.compile(**compile_args)

In [81]:
# Graph Visualize

# Visualize the graph structure
graph = build_research_graph()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	synthesiser(synthesiser)
	critic(critic)
	researcher_1(researcher_1<hr/><small><em>__interrupt = before</em></small>)
	researcher_2(researcher_2)
	researcher_3(researcher_3)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	critic -. &nbsp;end&nbsp; .-> __end__;
	critic -. &nbsp;revise&nbsp; .-> synthesiser;
	planner --> researcher_1;
	planner --> researcher_2;
	planner --> researcher_3;
	researcher_1 --> synthesiser;
	researcher_2 --> synthesiser;
	researcher_3 --> synthesiser;
	synthesiser --> critic;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [83]:
# Attempt to save as PNG
try:
    png_data = graph.get_graph().draw_mermaid_png()
    with open("graph_structure.png", "wb") as f:
        f.write(png_data)
    print("Graph visualization saved to graph_structure.png")
except Exception:
    print("Note: Could not generate PNG. Install graphviz system package to enable image generation.")

Graph visualization saved to graph_structure.png


### Single Agent /  BaseLine Agent

In [16]:
from langchain_core.messages import HumanMessage

BASELINE_SYSTEM_PROMPT = """You are an expert research assistant. Answer the user's question thoroughly and accurately.

CRITICAL RULE: Before answering, you MUST make separate searches for EACH major topic, entity, or aspect mentioned in the question. 
- For comparison questions: search for EACH entity separately.
- For "causes and responses" questions: search for causes separately from responses.
- For summary questions: search for each key sub-topic separately.
- Make ALL your searches in one step. Do not wait for results between searches.

RULES:
1. Cite sources using [1], [2], etc., matching the search results.
2. Do not guess or make up information. Only use what the search results provide.
3. Provide a detailed, well-structured answer with clear paragraphs."""

def run_baseline(question: str) -> dict:
    print("\n" + "=" * 70)
    print("BASELINE AGENT - Single Agent with Web Search")
    print("=" * 70)
    print(f"Question: {question}", flush=True)
    print("-" * 70, flush=True)

    tools_map = {t.name: t for t in [web_search, calculator]}

    start_time = time.time()
    tool_steps = 0

    # ---- STEP 1: Call with tools, let model decide to search ----
    print("  [Step 1] Calling LLM with tools...", flush=True)
    llm_with_tools = llm.bind_tools([web_search, calculator], parallel_tool_calls = True)
    
    for attempt in range(5):
        try:
            response = llm_with_tools.invoke([
                {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ])
            break
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e):
                wait = 20 * (attempt + 1)
                print(f"  [Limit hit] Waiting {wait}s...", flush=True)
                time.sleep(wait)
            else:
                raise e

    # ---- STEP 2: Execute tools, collect results ----
    all_results = ""
    if response.tool_calls:
        for tc in response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            print(f"  [Step 2] Tool: {tool_name}({tool_args})", flush=True)

            if tool_name in tools_map:
                try:
                    result = tools_map[tool_name].invoke(tool_args)
                    tool_steps += 1
                    if len(result) > 2000:
                        result = result[:2000] + "...[truncated]"
                except Exception as e:
                    result = f"Error: {str(e)}"
            else:
                result = f"Error: Tool '{tool_name}' not found."

            query = tool_args.get("query", "")
            all_results += f"\n[{tool_steps}] Search: {query}\n{result}\n"

        time.sleep(3)

    # ---- STEP 3: Call WITHOUT tools, just answer ----
    print("  [Step 3] Calling LLM for final answer...", flush=True)
    for attempt in range(5):
        try:
            final_response = llm.invoke([
                {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
                {"role": "user", "content": question},
                {"role": "user", "content": f"Here are the search results:\n{all_results}\n\nWrite a detailed answer with citations."},
            ])
            break
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e):
                wait = 20 * (attempt + 1)
                print(f"  [Limit hit] Waiting {wait}s...", flush=True)
                time.sleep(wait)
            else:
                raise e

    final_answer = final_response.content
    end_time = time.time()

    print(f"  [Done] Final answer ({len(final_answer)} chars)", flush=True)

    return {
        "question": question,
        "answer": final_answer,
        "time_seconds": round(end_time - start_time, 2),
        "steps": tool_steps
    }

In [16]:
test_questions = [
        "Compare the current EV strategies of Tata Motors and BYD.",
        # "What are the main causes and latest policy responses to urban air pollution in Delhi?",
        # "Summarize the 2024-2025 advances in small open-weight language models."
    ]

baseline_results = []

for q in test_questions:
    result = run_baseline(q)
    baseline_results.append(result)

    print("\n" + "-" * 70)
    print("ANSWER PREVIEW:")
    answer_text = result["answer"]
    print(answer_text[:15000] + "..." if len(answer_text) > 1500 else answer_text)
    print(f"\n Time: {result['time_seconds']}s | Tool Calls: {result['steps']}")

with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("\n" + "=" * 70)
print("BASELINE COMPLETE. Results saved to baseline_results.json")
print("=" * 70)


BASELINE AGENT - Single Agent with Web Search
Question: Compare the current EV strategies of Tata Motors and BYD.
----------------------------------------------------------------------
  [Step 1] Calling LLM with tools...
  [Step 2] Tool: web_search({'query': 'Tata Motors EV strategy 2024 BYD EV strategy 2024'})
  [Step 3] Calling LLM for final answer...
  [Done] Final answer (8029 chars)

----------------------------------------------------------------------
ANSWER PREVIEW:
**Tata Motors – Indian‑centric, platform‑driven expansion**

| Aspect | What Tata Motors is doing (2024) | Source |
|--------|----------------------------------|--------|
| **Market position** | Holds the dominant share of the Indian electric‑passenger‑vehicle market – **73.1 %** in FY 2024. | 【1†L1-L3】 |
| **Revenue from EVs** | FY 24 EV‑related revenue reached **₹9,300 cr** (≈ US$ 110 m). | 【1†L1-L3】 |
| **Strategic framework** | Launched a **“2‑2‑2” EV strategy** that aims to capture **50 %** of the Indian EV m

In [27]:
# def demonstrate_human_loop():
#     question = "Assess the investment case for green hydrogen in India."
    
#     # 1. Initialize checkpointer and build graph
#     memory = InMemorySaver()
#     graph = build_research_graph(checkpointer=memory)
    
#     thread_id = "human-demo-session-001"
#     config = {"configurable": {"thread_id": thread_id}}
    
#     initial_state = {
#         "question": question, "plan": [], "findings": [], "draft": "",
#         "references": [], "critique": "", "iterations": 0, "approved": False
#     }
    
#     # ========================================================================
#     # STAGE 1: RUN PLANNER AND PAUSE
#     # ========================================================================
#     print("=" * 70)
#     print("STAGE 1: RUNNING PLANNER")
#     print("=" * 70)
    
#     try:
#         graph.invoke(initial_state, config)
#     except GraphInterrupt:
#         pass # Silently catch the pause
    
#     # ========================================================================
#     # STAGE 2: INSPECT THE SAVED STATE (Cleanly)
#     # ========================================================================
#     print("\n" + "=" * 70)
#     print("STAGE 2: INSPECTING SAVED STATE")
#     print("=" * 70)
    
#     snapshot = memory.get(config)
#     current_plan = snapshot["channel_values"]["plan"]
    
#     print(f"\nPlanner generated {len(current_plan)} sub-questions:")
#     for i, q in enumerate(current_plan, 1):
#         print(f"  {i}. {q}")
        
#     # ========================================================================
#     # STAGE 3: REAL HUMAN INTERACTION
#     # ========================================================================
#     print("\n" + "=" * 70)
#     print("STAGE 3: HUMAN INTERVENTION")
#     print("=" * 70)
    
#     action = input("\nType 'yes' to approve, or 'edit' to add a question: ").strip().lower()
    
#     if action == 'edit':
#         new_question = input("Type the new sub-question to add: ").strip()
#         edited_plan = current_plan.copy()
#         edited_plan.append(new_question)
        
#         # THE CORRECT WAY: Update the paused state without restarting the graph
#         graph.update_state(config, {"plan": edited_plan})
#         print(f"\n[SYSTEM] Plan updated to {len(edited_plan)} questions. Resuming...")
#     else:
#         print("\n[SYSTEM] Plan approved. Resuming...")
        
#     # ========================================================================
#     # STAGE 4: RESUME GRAPH
#     # ========================================================================
#     print("\n" + "=" * 70)
#     print("STAGE 4: RESUMING GRAPH")
#     # ========================================================================
    
#     # Pass None to tell LangGraph "just resume where you left off"
#     final_state = graph.invoke(None, config)
    
#     print("\n" + "=" * 70)
#     print("SYSTEM COMPLETED")
#     print("=" * 70)
#     print(f"Final draft length: {len(final_state.get('draft', ''))} characters")
#     print(f"Critic iterations: {final_state.get('iterations', 0)}")

def demonstrate_human_loop():
    question = "Assess the investment case for green hydrogen in India."

    # 1. Initialize checkpointer and build graph
    memory = InMemorySaver()
    graph = build_research_graph(checkpointer=memory)

    thread_id = "human-demo-session-001"
    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "question": question,
        "plan": [],
        "findings": [],
        "draft": "",
        "references": [],
        "critique": "",
        "iterations": 0,
        "approved": False
    }

    # ============================================================
    # STAGE 1: RUN PLANNER AND PAUSE
    # ============================================================

    print("=" * 70)
    print("STAGE 1: RUNNING PLANNER")
    print("=" * 70)

    try:
        graph.invoke(initial_state, config)
    except Exception as e:
        print(f"[SYSTEM] Graph paused: {type(e).__name__}")

    # ============================================================
    # STAGE 2: INSPECT SAVED STATE
    # ============================================================

    print("\n" + "=" * 70)
    print("STAGE 2: INSPECTING SAVED STATE")
    print("=" * 70)

    snapshot = memory.get(config)

    current_plan = snapshot["channel_values"]["plan"]

    print(f"\nPlanner generated {len(current_plan)} sub-questions:")

    for i, q in enumerate(current_plan, 1):
        print(f"  {i}. {q}")

    # ============================================================
    # STAGE 3: HUMAN INTERVENTION
    # ============================================================

    print("\n" + "=" * 70)
    print("STAGE 3: HUMAN INTERVENTION")
    print("=" * 70)

    action = input(
        "\nType 'yes' to approve, or 'edit' to add a question: "
    ).strip().lower()

    if action == "edit":

        new_question = input(
            "Type the new sub-question to add: "
        ).strip()

        edited_plan = current_plan.copy()
        edited_plan.append(new_question)

        # Update the checkpointed state
        graph.update_state(
            config,
            {"plan": edited_plan}
        )

        print(
            f"\n[SYSTEM] Plan updated to "
            f"{len(edited_plan)} questions. Resuming..."
        )

    else:
        print("\n[SYSTEM] Plan approved. Resuming...")

    # ============================================================
    # STAGE 4: RESUME GRAPH
    # ============================================================

    print("\n" + "=" * 70)
    print("STAGE 4: RESUMING GRAPH")
    print("=" * 70)

    final_state = graph.invoke(None, config)

    # ============================================================
    # COMPLETED
    # ============================================================

    print("\n" + "=" * 70)
    print("SYSTEM COMPLETED")
    print("=" * 70)

    print(
        f"Final draft length: "
        f"{len(final_state.get('draft', ''))} characters"
    )

    print(
        f"Critic iterations: "
        f"{final_state.get('iterations', 0)}"
    )

In [29]:
# Run the demonstration
demonstrate_human_loop()

STAGE 1: RUNNING PLANNER
[PLANNER] Decomposing question: Assess the investment case for green hydrogen in India.
[PLANNER] Generated 5 sub-questions.

STAGE 2: INSPECTING SAVED STATE

Planner generated 5 sub-questions:
  1. What government policies and subsidies are currently available for green hydrogen production in India?
  2. What is the projected demand for green hydrogen in India's energy and industrial sectors by 2030?
  3. What are the current production costs and cost reduction pathways for green hydrogen in India compared to grey hydrogen?
  4. Which Indian companies and foreign investors have announced major green hydrogen projects or investments as of 2024?
  5. What infrastructure and logistical challenges affect the scaling of green hydrogen in India, such as storage, transport, and grid integration?

STAGE 3: HUMAN INTERVENTION

[SYSTEM] Plan approved. Resuming...

STAGE 4: RESUMING GRAPH
[RESEARCHER 1] Investigating: What government policies and subsidies are currently 

In [30]:
from langgraph.errors import GraphInterrupt

graph = build_research_graph(enable_interrupt=True)
state = {
    "question": "What is the current state of India's semiconductor manufacturing push?",
    "plan": [], "findings": [], "draft": "", "references": [],
    "critique": "", "iterations": 0, "approved": False
}
config = {"configurable": {"thread_id": "notebook-test"}}

# STEP 1: Run until interrupt
print("=== STAGE 1: Running Planner ===")
try:
    graph.invoke(state, config)
except GraphInterrupt:
    print("Graph paused for human review")

# STEP 2: Inspect the plan
snapshot = graph.get_state(config)
print(f"\nPlan generated: {snapshot.values['plan']}")

# STEP 3: Resume (this is the missing piece!)
print("\n=== STAGE 2: Resuming with Researchers ===")
final_state = graph.invoke(None, config)  # ← None means "continue from where you stopped"

print(f"Time taken: {round(time.time() - start, 2)}s")
print(f"Critic Iterations: {final_state['iterations']}")
print("\n--- MULTI-AGENT BRIEF ---")
print(final_state['draft'])

=== STAGE 1: Running Planner ===
[PLANNER] Decomposing question: What is the current state of India's semiconductor manufacturing push?
[PLANNER] Generated 5 sub-questions.

Plan generated: ['What government policies and financial incentives has India introduced to boost semiconductor manufacturing as of 2024?', 'Which semiconductor fabrication plants are currently operational or under construction in India, and what are their announced production capacities?', 'Which domestic and foreign companies are leading the semiconductor manufacturing initiatives in India?', "What are the primary challenges and bottlenecks facing India's semiconductor manufacturing sector, such as talent, supply chain, and funding constraints?", "What are the projected market size and economic contribution of India's semiconductor manufacturing industry by 2030?"]

=== STAGE 2: Resuming with Researchers ===
[RESEARCHER 1] Investigating: What government policies and financial incentives has India introduced to bo

In [12]:
from datetime import datetime

def create_initial_state(question: str) -> dict:
    """
    Creates the exact dictionary structure required by ResearchState.
    Every key must be present to satisfy the TypedDict contract.
    """
    return {
        "question": question,
        "plan": [],
        "findings": [],
        "draft": "",
        "references": [],
        "critique": "",
        "iterations": 0,
        "approved": False
    }


def sanitize_filename(question: str) -> str:
    """Converts a question string into a safe Windows/Mac filename."""
    safe_string = re.sub(r'[^\w\s-]', '', question)
    safe_string = re.sub(r'[-\s]+', '_', safe_string).strip("_")
    return safe_string[:50] # Truncate long names


def run_single_question(graph, question: str, question_id: int) -> dict:
    """Executes the graph for a single question and returns metrics."""
    print("\n" + "=" * 80)
    print(f"STARTING: {question}")
    print("=" * 80)
    
    start_time = time.time()
    initial_state = create_initial_state(question)
    config = {
        "configurable": {
            "thread_id": f"evaluation-question-{question_id}"
        }
    }
    
    # Invoke the graph. This triggers the entire Planner -> Researchers -> Synth -> Critic loop.
    final_state = graph.invoke(initial_state, config)
    
    end_time = time.time()
    duration = round(end_time - start_time, 2)
    
    return {
        "question": question,
        "draft": final_state.get("draft", "No draft generated."),
        "iterations": final_state.get("iterations", 0),
        "approved": final_state.get("approved", False),
        "time_seconds": duration,
        "timestamp": datetime.now().isoformat()
    }


def save_markdown(result: dict, output_dir: str = "outputs"):
    """Saves the research brief to a formatted Markdown file."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    filename = sanitize_filename(result["question"]) + ".md"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# Research Brief\n\n")
        f.write(f"**Question:** {result['question']}\n\n")
        f.write(f"**Generated:** {result['timestamp']}\n")
        f.write(f"**Critic Iterations:** {result['iterations']} | ")
        f.write(f"**Approved:** {'Yes' if result['approved'] else 'No (Max iterations reached)'} | ")
        f.write(f"**Time:** {result['time_seconds']}s\n\n")
        f.write("---\n\n")
        f.write(result["draft"])
        
    print(f"[SAVED] Output written to {filepath}")
    return filepath

In [28]:
evaluation_questions = [
        # "Compare the current EV strategies of Tata Motors and BYD.",
        # "What are the main causes and the latest policy responses to urban air pollution in Delhi?",
        # "Summarise the 2024-2025 advances in small open-weight language models and who is leading.",
        # "Assess the investment case for green hydrogen in India: drivers, risks, key players.",
         "How are major cloud providers pricing GPU compute in 2025, and how do they compare?",
        # "What is the current state of India's semiconductor manufacturing push?"
        # "Analyze the systemic, genetic, and socio-environmental drivers behind the epidemic of metabolic syndrome—specifically type 2 diabetes, hypertension, and secondary liver (MASLD), cardiac, and renal mortality—in the Indian population, contrasting the 'thin-fat' phenotype and dietary transition with Western epidemiological models, and propose an integrated, multi-tiered public health intervention framework for early screening and prevention."
    ]
    
print("Compiling Multi-Agent Graph...")
graph = build_research_graph(enable_interrupt=False)

all_results = []
total_start_time = time.time()

for i, question in enumerate(evaluation_questions, 1):
    print(f"\n\n{'#' * 80}")
    print(f"QUESTION {i} OF {len(evaluation_questions)}")
    print(f"{'#' * 80}")
    
    result = run_single_question(graph, question,i)
    save_markdown(result)
    all_results.append(result)
    
total_end_time = time.time()
total_duration = round(total_end_time - total_start_time, 2)

# Save all metrics to a JSON file for the evaluation step
with open("multiagent_metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)
    
print("\n" + "=" * 80)
print("EXECUTION COMPLETE")
print("=" * 80)
print(f"Total questions processed: {len(all_results)}")
print(f"Total execution time: {total_duration} seconds")
print(f"Average time per question: {round(total_duration / len(all_results), 2)} seconds")
print("Metrics saved to: multiagent_metrics.json")
print("Briefs saved to: /outputs/ directory")

Compiling Multi-Agent Graph...


################################################################################
QUESTION 1 OF 1
################################################################################

STARTING: How are major cloud providers pricing GPU compute in 2025, and how do they compare?
[PLANNER] Decomposing question: How are major cloud providers pricing GPU compute in 2025, and how do they compare?
[PLANNER] Generated 5 sub-questions.
[RESEARCHER 1] Investigating: What are the on-demand pricing rates for NVIDIA A100 GPU instances on Amazon Web Services (AWS) in 2025?
[RESEARCHER 2] Investigating: What are the on-demand pricing rates for NVIDIA A100 GPU instances on Microsoft Azure in 2025?
[RESEARCHER 3] Investigating: What are the on-demand pricing rates for NVIDIA A100 GPU instances on Google Cloud Platform (GCP) in 2025?
  [Tool] web_search(AWS on-demand pricing NVIDIA A100 2025)
  [Tool] web_search(Azure NVIDIA A100 GPU on-demand pricing 2025)
  [Tool] web_searc

In [16]:
JUDGE_PROMPT = """You are an impartial research evaluator. Score the following research brief on a scale of 1-5 for each criterion.

ORIGINAL QUESTION: {question}

RESEARCH BRIEF:
{brief}

EVALUATE EACH CRITERION (1=Terrible, 5=Excellent):
1. Coverage: Does the brief address ALL parts of the question?
2. Faithfulness: Is every claim supported by a cited source (no hallucinations)?
3. Citation Quality: Are citations real, relevant, and properly formatted?
4. Coherence: Is it well-structured and readable?

Respond ONLY in JSON format:
{{
    "coverage": <int>,
    "faithfulness": <int>,
    "citation_quality": <int>,
    "coherence": <int>,
    "average_score": <float>
}}"""

def evaluate_brief(question: str, brief: str) -> dict:
    """Uses an LLM to grade a brief out of 5."""
    if not brief or len(brief) < 100:
        return {"coverage": 1, "faithfulness": 1, "citation_quality": 1, "coherence": 1, "average_score": 1.0}
        
    response = llm.invoke([HumanMessage(content=JUDGE_PROMPT.format(question=question, brief=brief))])
    
    try:
        # Extract JSON from the response
        content = response.content
        if "```" in content:
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        return json.loads(content)
    except:
        return {"average_score": 3.0, "error": "Failed to parse evaluation"}

def run_evaluation():
    """Loads results, evaluates them, and prints the comparison table."""
    
    # Load data
    try:
        with open("./baseline_results.json", "r") as f:
            baseline_data = json.load(f)
    except FileNotFoundError:
        print("ERROR: baseline_results.json not found. Run baseline_agent.py first.")
        return

    try:
        with open("multiagent_metrics.json", "r") as f:
            multiagent_data = json.load(f)
    except FileNotFoundError:
        print("ERROR: multiagent_metrics.json not found. Run main.py first.")
        return

    results = []
    
    print("\n" + "=" * 90)
    print("EVALUATING BASELINE VS MULTI-AGENT SYSTEM")
    print("=" * 90)

    # Evaluate matching questions
    for b_res in baseline_data:
        question = b_res["question"]
        
        # Find matching multi-agent result
        m_res = next((m for m in multiagent_data if m["question"] == question), None)
        if not m_res:
            continue
            
        print(f"\nEvaluating: {question[:60]}...")
        
        b_eval = evaluate_brief(question, b_res["answer"])
        m_eval = evaluate_brief(question, m_res["draft"])
        
        row = {
            "topic": question[:45] + "...",
            "baseline_score": b_eval.get("average_score", 0),
            "multiagent_score": m_eval.get("average_score", 0),
            "baseline_time": b_res.get("time_seconds", 0),
            "multiagent_time": m_res.get("time_seconds", 0),
            "winner": "Multi-Agent" if m_eval.get("average_score", 0) > b_eval.get("average_score", 0) else "Baseline"
        }
        results.append(row)

    # Print Summary Table
    print("\n" + "-" * 90)
    print(f"{'TOPIC':<47} {'BASE':>6} {'MULTI':>6} {'TIME B':>7} {'TIME M':>7} {'WINNER':<12}")
    print("-" * 90)
    
    for r in results:
        print(f"{r['topic']:<47} {r['baseline_score']:>6.1f} {r['multiagent_score']:>6.1f} {r['baseline_time']:>6.0f}s {r['multiagent_time']:>6.0f}s {r['winner']:<12}")
        
    # Calculate Averages
    if results:
        avg_b_score = sum(r["baseline_score"] for r in results) / len(results)
        avg_m_score = sum(r["multiagent_score"] for r in results) / len(results)
        avg_b_time = sum(r["baseline_time"] for r in results) / len(results)
        avg_m_time = sum(r["multiagent_time"] for r in results) / len(results)
        
        print("-" * 90)
        print(f"{'AVERAGES':<47} {avg_b_score:>6.1f} {avg_m_score:>6.1f} {avg_b_time:>6.0f}s {avg_m_time:>6.0f}s")
        print(f"\nScore Improvement: +{avg_m_score - avg_b_score:.1f} points")
        print(f"Time Trade-off: +{avg_m_time - avg_b_time:.0f} seconds average")

    # Save evaluation results
    with open("evaluation_results.json", "w") as f:
        json.dump(results, f, indent=2)
        
    print("\nResults saved to evaluation_results.json")

In [ ]:
# Run the evaluation comparison
# Note: This requires baseline_results.json and multiagent_metrics.json to exist. 
# If you didn't run the full baseline earlier and multiagent_mterics, it will gracefully handle the missing file.
run_evaluation()


EVALUATING BASELINE VS MULTI-AGENT SYSTEM

Evaluating: Compare the current EV strategies of Tata Motors and BYD....

------------------------------------------------------------------------------------------
TOPIC                                             BASE  MULTI  TIME B  TIME M WINNER      
------------------------------------------------------------------------------------------
Compare the current EV strategies of Tata Mot...    2.8    4.8      7s      7s Multi-Agent 
------------------------------------------------------------------------------------------
AVERAGES                                           2.8    4.8      7s      7s

Score Improvement: +2.0 points
Time Trade-off: +-1 seconds average

Results saved to evaluation_results.json
